# P2 — Making code decide and repeat

**Python primer · session 2 of 4**


## What this session is for

Last week the notebook was a calculator: you typed an expression and it
answered. This week it starts to be a program.

Two ideas do that. **Deciding** — do one thing if a condition holds and
another if it does not. **Repeating** — do the same thing to many values
without typing it many times. Then we package both into **functions**, so that
a piece of work can be given a name and reused.

Everything is built on the same running example: three detectors, their state
and whether a candidate signal is worth looking at.


## Conditions: things that are true or false

A comparison produces a `bool`, which is either `True` or `False`.


In [ ]:
snr = 13.2          # signal to noise ratio of a candidate
threshold = 8.0

print(snr > threshold)
print(snr == 8.0)          # note: two equals signs asks a question
print(snr != 8.0)
print(type(snr > threshold))


One equals sign stores a value, two equals signs ask whether two values are
equal. Confusing them is the most common beginner error in every language that
has both.

Conditions combine with `and`, `or` and `not`:


In [ ]:
h1_observing = True
l1_observing = True
v1_observing = False

print(h1_observing and l1_observing)      # both
print(v1_observing or l1_observing)       # at least one
print(not v1_observing)                   # the opposite

# a real question: can we localise this event on the sky?
print("three detector event:", h1_observing and l1_observing and v1_observing)


## Deciding: `if`, `elif`, `else`

The indentation is not decoration. In Python the indented block **is** the body
of the `if`; there are no braces. Four spaces, consistently.


In [ ]:
snr = 13.2

if snr > 12:
    print("loud candidate: alert the astronomers")
elif snr > 8:
    print("candidate: run the follow-up checks")
else:
    print("below threshold: this is noise")


Read it as a chain: the first condition that is `True` wins and the rest are
skipped. `elif` is short for "else if" and `else` catches everything left.

> **Careful — A colon and an indent, every time**
>
> Every line that opens a block ends with a colon and everything inside is
> indented. Forget the colon and you get `SyntaxError`; forget the indent and
> you get `IndentationError`.
>
> Both messages name the line. Read them rather than guessing.

Change `snr` in the cell above to `9.0` and to `3.0` and run it again. Watch
which branch fires.


## Repeating: the `for` loop

A `for` loop walks through a collection, one item at a time.


In [ ]:
detectors = ["H1", "L1", "V1"]

for name in detectors:
    print("checking", name)


`name` is a variable that takes each value in turn. The body runs once per
item.

Loops and decisions combine, which is where programs start doing something
useful:


In [ ]:
detectors = ["H1", "L1", "V1", "K1"]
ranges_mpc = [160.0, 155.0, 55.0, 6.0]     # BNS range of each, in megaparsecs

for name, r in zip(detectors, ranges_mpc):
    if r > 100:
        role = "carries the detection"
    elif r > 20:
        role = "helps localise the source"
    else:
        role = "contributes little at this sensitivity"
    print(f"{name}: {r:6.1f} Mpc  -- {role}")


Two new things in that cell, both worth keeping.

`zip(a, b)` walks two lists together, handing you one item from each.

`f"{name}: {r:6.1f} Mpc"` is an **f-string**: put an `f` before the quotes and
anything in braces is evaluated and inserted. `{r:6.1f}` means *six characters
wide, one digit after the point*, which is how the numbers above line up in a
column. You will use this constantly.


### Counting with `range`

When you need to repeat a fixed number of times rather than walk a list:


In [ ]:
for i in range(5):
    print(i, "->", 2 ** i)


`range(5)` gives $0, 1, 2, 3, 4$ — it starts at zero and stops *before* the
number you give. Python counts from zero everywhere and this is the single
most common source of off-by-one mistakes for people arriving from other
backgrounds.

> **Careful — A loop you should learn not to write**
>
> You can loop over data samples like this:
>
> ```python
> for i in range(len(strain)):
>     result[i] = strain[i] * 2

It works and for the four thousand samples per second of real strain data it
is slow enough to matter. Next session shows the one line that replaces it and
runs about a hundred times faster.

Recognise this shape and be suspicious of it. That habit is the whole of P3.
```


### Repeating until something happens: `while`

A `for` loop repeats a known number of times. A `while` loop repeats until a
condition stops being true, which is what you want when you do not know in
advance how many steps it will take.


In [ ]:
# how long does a binary take to sweep from 30 Hz to 300 Hz?
# a crude model: the frequency grows by 8 per cent each step of 0.1 s
f = 30.0
t = 0.0
while f < 300.0:
    f = f * 1.08
    t = t + 0.1

print(f"reached {f:.0f} Hz after {t:.1f} s")


> **Careful — Every while loop needs an exit**
>
> If the condition never becomes false the loop never ends and the notebook
> hangs with the cell marked `[*]` forever. Interrupt it with the stop button.
>
> Before writing a `while`, say out loud what makes it stop. In the cell above it
> is that `f` grows every pass. Take away the growth and it runs until the heat
> death of the universe.

Two words change the flow inside any loop. `break` leaves it immediately;
`continue` skips to the next pass.


In [ ]:
snrs = [4.1, 9.7, 6.2, 13.2, 7.9, 24.4, 5.5]

for s in snrs:
    if s < 8:
        continue                 # not interesting, go to the next
    print("candidate:", s)
    if s > 20:
        print("  very loud -- stopping the scan here")
        break                    # leave the loop altogether


Use them sparingly. A loop with three `break`s in it is usually a loop that
wanted to be a function.


## Functions: naming a piece of work

A function is a named block of code that takes inputs and gives back a result.
You have been calling them — `print`, `type`, `len`. Now you define one.


In [ ]:
def strain_to_displacement(h, arm_length):
    """Return the change in arm length, in metres, for a strain h."""
    return h * arm_length


print(strain_to_displacement(1e-21, 4000))
print(strain_to_displacement(1e-21, 3000))


The pieces: `def`, the name, the **parameters** in brackets, a colon, the
indented body and `return` to hand a value back.

The string on the first line is a **docstring**. It is not a comment: Python
keeps it and `help()` shows it. Write one for every function, saying what
comes back and in what units.


In [ ]:
help(strain_to_displacement)


### Defaults and calling by name

A parameter can have a default value, which makes it optional:


In [ ]:
def displacement(h, arm_length=4000.0):
    """Change in arm length in metres. Default arm is LIGO's 4 km."""
    return h * arm_length


print(displacement(1e-21))                      # uses the default
print(displacement(1e-21, 3000.0))              # by position
print(displacement(1e-21, arm_length=3000.0))   # by name -- clearest


> **Important — Call by name whenever there is more than one argument**
>
> `bandpass(data, 4096, 35, 350)` is a puzzle. `bandpass(data, fs=4096,
> low=35, high=350)` is a sentence.
>
> Every library you will meet this term — `numpy`, `scipy`, `gwpy`, `bilby` —
> has functions with many optional arguments. Reading their documentation and
> calling them by name is the skill that carries you through the course, far
> more than writing your own functions.


### What a function can and cannot see

Names created inside a function live only there. This is a feature: it means
you can write a function without worrying about what else is in the notebook.


Names created inside a function are **local** to it. A name defined outside is
**global** and can be read from inside, but not changed by assignment:


In [ ]:
sample_rate = 4096          # a global name

def report():
    resolution = 1 / sample_rate         # reading it works
    return f"one sample is {resolution * 1000:.3f} ms"

print(report())


Now try to change it from inside.


In [ ]:
def broken():
    sample_rate = 16384        # this makes a NEW local name
    return sample_rate

print(broken(), "inside")
print(sample_rate, "outside -- unchanged")


That last pair of lines is the rule: assigning inside a function creates a
local name and leaves the outer one alone. It looks surprising once and then
saves you constantly, because it means a function cannot quietly break the rest
of your notebook.

Python does have a `global` keyword to override this. Almost every use of it is
a mistake — pass the value in as an argument and return the new one instead.


In [ ]:
def noisy_example():
    inside = 42          # exists only during this call
    return inside * 2


print(noisy_example())

try:
    print(inside)
except NameError as e:
    print("NameError:", e)


The `try` / `except` in that cell is the other half of last week's lesson on
errors. It says: attempt this and if that particular error happens, do
something else instead of stopping.

Use it sparingly. An error you did not expect should stop the program, loudly —
hiding it makes a wrong result look like a right one.


## Exercises


### 1. Which detectors were observing

Given the lists below, print one line per detector saying whether it was
observing, using a `for` loop and an `if`.


In [ ]:
detectors = ["H1", "L1", "V1"]
observing = [True, True, False]

# your code here


> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


### 2. A function with a default

Write `time_delay(distance_km, speed=299792.458)` returning the light travel
time in seconds, with a docstring. Use it for the 3000 km between the two LIGO
sites and compare with the 7 ms delay measured in GW150914.


In [ ]:
# your code here


> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


### 3. Count the candidates

Given a list of signal to noise ratios, count how many are above 8 and print
the loudest. Do it with a loop, an `if` and a counter.


In [ ]:
snrs = [4.1, 9.7, 6.2, 13.2, 7.9, 24.4, 5.5]

# your code here


> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


## Before the next session

**Do.** `reference/modulo_2_control_flow/exercises/` and
`reference/modulo_3_functions/exercises/`, with solutions beside them.

**Check.** You should be able to write a `for` loop over two lists at once,
explain what `range(5)` produces and say why a docstring is not a comment.

**Next session is the important one.** Arrays and the habit of operating on a
whole time series at once instead of one sample at a time. Everything you do
with real data for the rest of the course rests on it.
